#  Phase 1  Data Cleaning & Feature Engineering
### EEIA 2025 : Analyse de Survie & Médecine de Précision

---

##  Objectifs de cette phase

À la fin de cette phase, vous serez capables de :
- Charger et inspecter un jeu de données cliniques réel
- Comprendre les variables de survie (`temps` + `événement`)
- Nettoyer les données : valeurs manquantes, types, encodage
- Créer de nouvelles variables utiles (feature engineering)

---

##  Installation des dépendances

In [ ]:
# Si vous êtes sur Google Colab, décommentez la ligne suivante :
# !pip install lifelines --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

print(' Dépendances chargées avec succès')

## Chargement des données

Les données proviennent de cBioPortal : **MSK-IMPACT Breast Cancer 2025**

> 🔗 https://www.cbioportal.org/study/clinicalData?id=breast_msk_2025

Le fichier est au format TSV (Tab-Separated Values).

In [ ]:
# Chargement du fichier
# Adaptez le chemin selon votre environnement
DATA_PATH = "../../data/clinical_data.tsv"

df_raw = pd.read_csv(DATA_PATH, sep="\t", low_memory=False)

print(f"Dimensions : {df_raw.shape[0]} lignes × {df_raw.shape[1]} colonnes")
df_raw.head(3)

##  Exploration initiale

In [ ]:
# Afficher tous les noms de colonnes
print("Colonnes disponibles :")
for col in df_raw.columns:
    print(f"  - {col}")

In [ ]:
# Types de données et valeurs manquantes
info = pd.DataFrame({
    'Type': df_raw.dtypes,
    'Non-null': df_raw.count(),
    'Manquants': df_raw.isnull().sum(),
    '% Manquants': (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
})
info[info['Manquants'] > 0].sort_values('% Manquants', ascending=False)

---

##  Concepts clés : Les données de survie

En analyse de survie, on a besoin de **deux variables** pour chaque patient :

1. **T** — le temps écoulé jusqu'à l'événement (ou jusqu'à la fin de l'étude)
2. **E** — indicateur de l'événement (1 = décès observé, 0 = censuré)

### Qu'est-ce que la censure ?

Un patient est **censuré** quand on ne sait pas s'il est décédé parce que :
- L'étude s'est terminée avant son décès
- Il a quitté le suivi (perdu de vue)

 On **NE peut pas** ignorer ces patients — ils contiennent de l'information !

```
Patient A : ──────●  (décédé à 24 mois → E=1, T=24)
Patient B : ────────────○  (suivi jusqu'à 36 mois, vivant → E=0, T=36)
Patient C : ──────○  (perdu de vue à 18 mois → E=0, T=18)
```

---

##  Exercice 1 : Parser la variable cible

La colonne `Overall Survival Status` contient des chaînes comme `'1:DECEASED'` ou `'0:LIVING'`.

**Tâche :** Créez une colonne `event` avec :
- `1` si le patient est décédé (`DECEASED`)
- `0` si le patient est vivant ou censuré (`LIVING`)

In [ ]:
df = df_raw.copy()

# Vérifiez d'abord les valeurs uniques de la colonne
print(df['Overall Survival Status'].value_counts())

In [ ]:
#  VOTRE CODE ICI
# Créez la colonne 'event'

# df['event'] = ...

# Vérification
# print(df['event'].value_counts())
# print(f"Taux d'événements : {df['event'].mean():.1%}")

##  Exercice 2 : Nettoyer la variable temporelle

**Tâche :**
1. Convertissez `Overall Survival (Months)` en numérique
2. Affichez les statistiques descriptives
3. Combien de valeurs manquantes y a-t-il ?

In [ ]:
#  VOTRE CODE ICI

# df['Overall Survival (Months)'] = ...
# print(...)

##  Exercice 3 : Gestion des valeurs manquantes

**Tâche :**
1. Pour `Metastatic Site` : imputer les valeurs manquantes par `'Unknown'`
2. Pour `Tumor Purity` : regarder la distribution et décider d'une stratégie (imputation par médiane OU suppression des lignes)
3. Pour `Mutation Count` : convertir en numérique

Justifiez votre choix pour `Tumor Purity`.

In [ ]:
#  VOTRE CODE ICI

# 1. Metastatic Site
# df['Metastatic Site'] = ...

# 2. Tumor Purity — analysez d'abord !
# print(df['Tumor Purity'].describe())
# print(f"Manquants : {df['Tumor Purity'].isnull().sum()}")
# Votre stratégie : ...

# 3. Mutation Count
# df['Mutation Count'] = ...

## Exercice 4 : Feature Engineering

**Tâche :** Créez deux nouvelles variables :

1. `is_metastatic` : 1 si `Sample Type == 'Metastasis'`, 0 sinon
2. `FGA_high` : 1 si `Fraction Genome Altered >= 0.2`, 0 sinon

Ces variables seront utilisées dans les phases suivantes.

In [ ]:
#  VOTRE CODE ICI

# df['is_metastatic'] = ...
# df['FGA_high'] = ...

# Vérification des nouvelles colonnes
# print(df[['Sample Type', 'is_metastatic']].value_counts())
# print(df[['Fraction Genome Altered', 'FGA_high']].describe())

##  Vérification finale

Votre dataset nettoyé doit contenir :
- La colonne `event` (0 ou 1)
- La colonne `Overall Survival (Months)` en numérique
- Les colonnes `is_metastatic` et `FGA_high`
- Aucune valeur manquante dans ces colonnes clés

In [ ]:
# Vérification
key_cols = ['Overall Survival (Months)', 'event', 'Mutation Count',
            'Fraction Genome Altered', 'is_metastatic', 'FGA_high']

# Colonnes existantes seulement
existing = [c for c in key_cols if c in df.columns]
print("Colonnes clés, valeurs manquantes :")
print(df[existing].isnull().sum())

print(f"\n Dataset final : {df.shape[0]} patients × {df.shape[1]} variables")

## 💾 Sauvegarde pour la Phase 2

In [ ]:
df.to_csv("../../data/cleaned_data.csv", index=False)
print(" Données sauvegardées → data/cleaned_data.csv")

---

## 🧠 Questions de réflexion

1. Pourquoi ne peut-on pas supprimer tous les patients censurés ?
2. Quelle est la proportion de décès dans ce dataset ? Est-ce représentatif ?
3. Quels biais pourrait introduire l'imputation par médiane de `Tumor Purity` ?

---

**Suite : Phase 2  Analyse Exploratoire & Courbes de Kaplan-Meier**